# 01 — Inspect regional dataset and checkpoints (Phase 2)

**Purpose.** Read-only verification of the inputs before any new experiment:

1. discover the attached inputs under `/kaggle/input/` (no hardcoded account paths, no API token);
2. validate the regional ERA5 (WeatherBench 5.625°) dataset: splits, shapes, variables, coordinates, timestamps, normalisation, NaN/inf, grid padding;
3. cross-check the Kaggle copy against the MD5 manifest of the local dataset (`docs/mtech/local_dataset_manifest.json`);
4. inspect every checkpoint: training state, hyper-parameters, callback state, inferred architecture;
5. compare checkpoint metadata with the internship training log to judge provenance;
6. strict-load a checkpoint and run a **small forward-pass smoke test** (a few dozen samples, *not* an evaluation).

**Attach (Add Input):** `agrawaladi/climax-indian-monsoon-regional` and `agrawaladi/checkpoints-for-regional-forecasting`.
The repository code is taken from an attached copy of the repo if present, otherwise cloned from GitHub (`REPO_BRANCH`).

Outputs go to `/kaggle/working/inspection/`. Inputs are never modified; padding (if needed for the 6×6 model) is applied in memory only.

In [ ]:
# ---- Configuration: edit only if auto-discovery is ambiguous ----
INPUT_ROOT = "/kaggle/input"
DATA_DIR_OVERRIDE = None   # e.g. "/kaggle/input/.../regional_climax_dataset"
CKPT_DIR_OVERRIDE = None   # directory that contains the *.ckpt files
REPO_DIR_OVERRIDE = None   # attached copy of the repo (must contain src/climax and src/climax_hazards)
REPO_URL = "https://github.com/Aditya1Agrawal/ClimaX-Monsoon-PINN.git"
REPO_BRANCH = "mtech-extreme-events"
OUT_DIR = "/kaggle/working/inspection"

INSTALL_DEPS = True        # installs timm==0.6.12 (the version the internship run used)
RUN_MD5 = True             # ~1 GB of hashing, about a minute
LOAD_CHECKPOINTS = True
RUN_FORWARD_TEST = True
FORWARD_CKPT_NAME = "epoch_016.ckpt"   # checkpoint used for the smoke test (falls back to the first one found)
FORWARD_N_SAMPLES = 64
SEED = 42

In [ ]:
import os, sys, json, subprocess, pathlib, platform
os.makedirs(OUT_DIR, exist_ok=True)
print("Python", platform.python_version())

def show_tree(root, max_files=6, max_depth=5):
    root = pathlib.Path(root)
    for dirpath, dirnames, filenames in os.walk(root):
        rel = pathlib.Path(dirpath).relative_to(root)
        if len(rel.parts) > max_depth:
            dirnames[:] = []
            continue
        dirnames.sort()
        pad = "  " * len(rel.parts)
        print(f"{pad}{pathlib.Path(dirpath).name}/ ({len(filenames)} files)")
        for f in sorted(filenames)[:max_files]:
            print(f"{pad}  {f}  [{os.path.getsize(os.path.join(dirpath, f)) / 2**20:.1f} MB]")
        if len(filenames) > max_files:
            print(f"{pad}  ... {len(filenames) - max_files} more")

show_tree(INPUT_ROOT)

## Repository code

In [ ]:
def find_repo(root):
    for dirpath, dirnames, _ in os.walk(root):
        if os.path.isdir(os.path.join(dirpath, "src", "climax_hazards")) and os.path.isdir(os.path.join(dirpath, "src", "climax")):
            return dirpath
    return None

REPO_DIR = REPO_DIR_OVERRIDE or find_repo(INPUT_ROOT)
if REPO_DIR is None:
    REPO_DIR = "/kaggle/working/ClimaX-Monsoon-PINN"
    if not os.path.isdir(os.path.join(REPO_DIR, "src", "climax_hazards")):
        r = subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR], capture_output=True, text=True)
        print(r.stdout[-2000:], r.stderr[-2000:])
        if r.returncode != 0:
            raise RuntimeError(
                f"Could not clone branch {REPO_BRANCH!r}. Either push the branch to GitHub, or upload the repo "
                "as a Kaggle dataset, attach it, and set REPO_DIR_OVERRIDE.")
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
print("REPO_DIR:", REPO_DIR)
r = subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--format=%H %s"], capture_output=True, text=True)
print("commit:", r.stdout.strip() or "(not a git checkout)")

In [ ]:
if INSTALL_DEPS:
    import importlib.util
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "timm==0.6.12"], check=True)
    timm_init = pathlib.Path(importlib.util.find_spec("timm").origin).parent / "models" / "__init__.py"
    txt = timm_init.read_text()
    # timm 0.6.12's maxxvit module fails to import on Python >= 3.11 (dataclass mutable default).
    # ClimaX does not use it, so the import is disabled (same patch as the internship notebook).
    if sys.version_info >= (3, 11) and "\nfrom .maxxvit import *" in txt:
        timm_init.write_text(txt.replace("\nfrom .maxxvit import *", "\n# from .maxxvit import *  # disabled for py>=3.11"))
import timm, torch, numpy as np
print("timm", timm.__version__, "| torch", torch.__version__, "| numpy", np.__version__, "| cuda", torch.cuda.is_available())

## Discover the attached inputs

In [ ]:
from climax_hazards import inspect_utils as iu

data_candidates = [p for p in iu.find_dataset_dirs(INPUT_ROOT) if "fake_era5_demo" not in str(p)]
DATA_DIR = iu.resolve_single(data_candidates, DATA_DIR_OVERRIDE, "regional dataset")
ckpt_root = CKPT_DIR_OVERRIDE or INPUT_ROOT
CKPTS = [p for p in iu.find_checkpoints(ckpt_root) if not str(p).startswith(str(REPO_DIR))]
print("DATA_DIR:", DATA_DIR)
print("Checkpoints:")
for p in CKPTS:
    print(f"  {p}  [{iu.human_bytes(p.stat().st_size)}]")
if not CKPTS:
    print("  none found - attach checkpoints-for-regional-forecasting or set CKPT_DIR_OVERRIDE")

## Dataset summary and integrity checks

In [ ]:
summary = iu.summarize_dataset(DATA_DIR, check_values=True)
np.set_printoptions(linewidth=160)
print("lat (ascending=%s):" % summary["lat_ascending"], summary["lat"])
print("lon:", summary["lon"])
for split, info in summary["splits"].items():
    print(f"\n[{split}] shards={info['n_shards']} years={info['years'][0]}-{info['years'][-1]} "
          f"steps={info['n_time_steps']} first={info.get('first_time_utc')} last={info.get('last_time_utc')}")
    print("  variables:", info["variables"])
    print("  shapes:", sorted({str(s) for v in info["shapes"].values() for s in v}))
    print("  non-finite:", info["nonfinite"] or "none")
print("\nZero-std variables (cannot be normalised):", summary["zero_std_variables"])
print("\nNormalisation (train-only statistics):")
for k in summary["normalize_mean"]:
    print(f"  {k:28s} mean={summary['normalize_mean'][k][0]:>12.3f}  std={summary['normalize_std'][k][0]:>10.3f}")
print("\nISSUES:", summary["issues"] or "none")
json.dump(summary, open(os.path.join(OUT_DIR, "dataset_summary.json"), "w"), indent=1, default=str)

### Grid and padding

The internship notebook copied the dataset to `/kaggle/working` and padded the longitude axis from 5 to 6 columns with `np.pad(..., mode="edge")`, then appended `lon[-1] + 5.625`.
The check below tells us whether the **attached** files are already padded (they should not be). Never pad a 6-column array again.

In [ ]:
pad = summary["padding"]
print(json.dumps(pad, indent=1))
if pad["looks_padded"]:
    print("Attached data already has a duplicated last column -> do NOT pad again.")
elif pad["n_lon"] == 5:
    print("Attached data is the unpadded 6x5 grid. The internship model (img_size [6, 6]) needs the in-memory edge pad;"
          " the extra column is a copy of lon=95.625 labelled 101.25 (not real data).")

In [ ]:
if RUN_MD5:
    manifest_path = os.path.join(REPO_DIR, "docs", "mtech", "local_dataset_manifest.json")
    ref = json.load(open(manifest_path))["files"]
    actual = iu.md5_manifest(DATA_DIR)
    cmp = iu.compare_manifests(actual, ref)
    print({k: (v if not isinstance(v, list) else v[:10]) for k, v in cmp.items()})
    print("Kaggle dataset identical to the local dataset:" , cmp["identical"])

## Checkpoints

Metadata only (one checkpoint in memory at a time, memory-mapped). Checkpoints are loaded with `weights_only=False` because Lightning
stores hyper-parameters as Python objects; this is safe only because these are your own files.

In [ ]:
ckpt_infos = {}
if LOAD_CHECKPOINTS:
    for p in CKPTS:
        info = iu.summarize_checkpoint(p)
        ckpt_infos[p.name] = info
        hp, dhp = info["hyper_parameters"], info["datamodule_hyper_parameters"]
        mc = {k: v for k, v in info["callbacks"].items() if k.startswith("ModelCheckpoint")}
        es = {k: v for k, v in info["callbacks"].items() if k.startswith("EarlyStopping")}
        print(f"\n=== {p.name} ({info['size']}) ===")
        print("  epoch:", info.get("epoch"), "| global_step:", info.get("global_step"), "| PL:", info.get("pytorch-lightning_version"))
        print("  optimizer states:", info["has_optimizer_states"], "| params:", f"{info['n_params']/1e6:.2f}M", "| non-net keys:", info["non_net_keys"])
        print("  inferred arch:", info["inferred_arch"])
        print("  model hparams:", {k: hp.get(k) for k in ("pretrained_path", "lr", "physics_lambda", "physics_geopotential_upper",
                                                         "physics_geopotential_lower", "physics_temperature_upper", "physics_temperature_lower",
                                                         "warmup_epochs", "max_epochs")})
        print("  data hparams:", {k: dhp.get(k) for k in ("root_dir", "predict_range", "hrs_each_step", "batch_size")},
              "| n variables:", len(dhp.get("variables", []) or []))
        for k, v in mc.items():
            print("  ", k[:80], "->", {kk: v.get(kk) for kk in ("best_model_score", "best_model_path", "best_k_models", "last_model_path")})
        for k, v in es.items():
            print("  ", k[:80], "->", {kk: v.get(kk) for kk in ("wait_count", "best_score", "stopped_epoch")})
    json.dump(ckpt_infos, open(os.path.join(OUT_DIR, "checkpoint_summary.json"), "w"), indent=1, default=str)

### Provenance check against the internship log

The internship notebook log (physics run, `physics_lambda = 0.001`, 2×GPU DDP, 4092 optimiser steps per epoch) reported:
`val/w_rmse` = 103.62589 at epoch 11 (step 49104), 103.58346 at epoch 16 (step 69564, best), 103.68472 at epoch 17 (step 73656); training stopped at `max_epochs=20`
and the test used `epoch_016.ckpt`. The table compares the checkpoint metadata with these logged values. A match is strong evidence, not proof, that a
file came from that run. No baseline (no-physics) checkpoint is known.

In [ ]:
EXPECTED = {
    "epoch_011.ckpt": {"epoch": 11, "global_step": 49104, "score": 103.62589},
    "epoch_016.ckpt": {"epoch": 16, "global_step": 69564, "score": 103.58346},
    "epoch_017.ckpt": {"epoch": 17, "global_step": 73656, "score": 103.68472},
    "last.ckpt": {"epoch": 19, "global_step": 81840, "score": None},
}
rows = []
for name, info in ckpt_infos.items():
    exp = EXPECTED.get(name, {})
    hp = info["hyper_parameters"]
    rows.append({
        "file": name, "epoch": info.get("epoch"), "exp_epoch": exp.get("epoch"),
        "global_step": info.get("global_step"), "exp_step": exp.get("global_step"),
        "physics_lambda": hp.get("physics_lambda"), "pretrained_path": repr(hp.get("pretrained_path")),
        "match": exp != {} and info.get("epoch") == exp.get("epoch") and info.get("global_step") == exp.get("global_step"),
    })
import pandas as pd
prov = pd.DataFrame(rows)
print(prov.to_string(index=False))
prov.to_csv(os.path.join(OUT_DIR, "checkpoint_provenance.csv"), index=False)

## Strict load and forward-pass smoke test

Builds `climax.arch.ClimaX` with the architecture in `configs/regional_india.yaml`, checks it against the shapes inferred from the checkpoint, compares keys
**before** loading, and loads with `strict=True` only if nothing is missing, unexpected or mis-shaped. Then runs a 72 h forecast on a few validation
samples and reports latitude-weighted RMSE next to persistence. This is a pipeline check on a handful of samples, **not** an evaluation result.

In [ ]:
import yaml
cfg = yaml.safe_load(open(os.path.join(REPO_DIR, "configs", "regional_india.yaml")))
net_args = cfg["model"]["net"]["init_args"]
VARS = cfg["data"]["variables"]
OUT_VARS = cfg["data"]["out_variables"]
PRED_RANGE = cfg["data"]["predict_range"] * cfg["data"]["hrs_each_step"]
print("config img_size:", net_args["img_size"], "patch:", net_args["patch_size"], "| vars:", len(VARS), "| lead:", PRED_RANGE, "h")

load_report = None
if RUN_FORWARD_TEST and CKPTS:
    from climax.arch import ClimaX
    from climax.utils.metrics import lat_weighted_mse
    torch.manual_seed(SEED)
    ck_path = next((p for p in CKPTS if p.name == FORWARD_CKPT_NAME), CKPTS[0])
    ckpt = iu.load_checkpoint(ck_path)
    sd = ckpt["state_dict"]
    inferred = iu.infer_climax_config(sd)
    if list(net_args["default_vars"]) != list(VARS):
        raise RuntimeError("config default_vars differ from data.variables; the variable order would be ambiguous")
    model = ClimaX(**net_args)
    mismatch = {k: (net_args.get(k), inferred.get(k)) for k in ("patch_size", "embed_dim", "depth", "decoder_depth")
                if net_args.get(k) != inferred.get(k)}
    if inferred["n_default_vars"] != len(VARS):
        mismatch["n_default_vars"] = (len(VARS), inferred["n_default_vars"])
    load_report = iu.compare_state_dict(model, sd)
    print("checkpoint:", ck_path.name)
    print("config vs checkpoint mismatches:", mismatch or "none")
    print("missing:", load_report["missing_in_checkpoint"])
    print("unexpected:", load_report["unexpected_in_checkpoint"])
    print("shape mismatch:", load_report["shape_mismatch"])
    if mismatch or any(load_report.values()):
        raise RuntimeError("Checkpoint is not compatible with configs/regional_india.yaml - see the report above. Not loading.")
    model.load_state_dict({k[len("net."):]: v for k, v in sd.items() if k.startswith("net.")}, strict=True)
    del ckpt, sd
    print("strict load OK")

In [ ]:
if RUN_FORWARD_TEST and CKPTS:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device).eval()
    lat = np.load(DATA_DIR / "lat.npy")
    mean_d, std_d = dict(np.load(DATA_DIR / "normalize_mean.npz")), dict(np.load(DATA_DIR / "normalize_std.npz"))
    mu = np.concatenate([mean_d[v] for v in VARS])[None, :, None, None]
    sd_ = np.concatenate([std_d[v] for v in VARS])[None, :, None, None]
    out_idx = [VARS.index(v) for v in OUT_VARS]

    files = iu.shard_files(DATA_DIR, "val")
    rng = np.random.default_rng(SEED)
    per_file = max(1, FORWARD_N_SAMPLES // len(files))
    X, Y = [], []
    for f in files:
        with np.load(f) as d:
            arr = np.concatenate([d[v].astype(np.float32) for v in VARS], axis=1)   # (T, V, H, W)
        t0 = rng.choice(arr.shape[0] - PRED_RANGE, size=per_file, replace=False)
        X.append(arr[t0]); Y.append(arr[t0 + PRED_RANGE])
    X, Y = np.concatenate(X)[:FORWARD_N_SAMPLES], np.concatenate(Y)[:FORWARD_N_SAMPLES]
    W_model = net_args["img_size"][1]
    padded = X.shape[-1] == W_model - 1
    if padded:   # replicate the internship preprocessing in memory only
        X, Y = iu.pad_lon_edge(X, W_model - 1), iu.pad_lon_edge(Y, W_model - 1)
    assert X.shape[-2:] == tuple(net_args["img_size"]), (X.shape, net_args["img_size"])

    xn = torch.from_numpy((X - mu) / sd_).float().to(device)
    lead = torch.full((len(xn),), PRED_RANGE / 100.0, device=device)   # same lead-time encoding as climax Forecast
    with torch.no_grad():
        preds = []
        for i in range(0, len(xn), 16):
            _, p = model(xn[i:i+16], xn[i:i+16][:, out_idx], lead[i:i+16], VARS, OUT_VARS, [lat_weighted_mse], lat=lat)
            preds.append(p.cpu().numpy())
    P = np.concatenate(preds) * sd_[:, out_idx] + mu[:, out_idx]
    Yo, Xo = Y[:, out_idx], X[:, out_idx]
    print(f"device={device} samples={len(P)} pred shape={P.shape} finite={np.isfinite(P).all()}")

    real = slice(0, X.shape[-1] - 1) if padded else slice(None)
    r_all = iu.lat_weighted_rmse_np(P, Yo, lat)
    r_real = iu.lat_weighted_rmse_np(P[..., real], Yo[..., real], lat)
    r_pers = iu.lat_weighted_rmse_np(Xo[..., real], Yo[..., real], lat)
    smoke = pd.DataFrame({"var": OUT_VARS, "rmse_6x6_incl_dup_col": r_all, "rmse_real_cols": r_real, "persistence_real_cols": r_pers})
    print("SMOKE TEST ONLY (few samples, val split) - not an evaluation result")
    print(smoke.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
    smoke.to_csv(os.path.join(OUT_DIR, "forward_smoke_test.csv"), index=False)

In [ ]:
report = {
    "data_dir": str(DATA_DIR),
    "checkpoints": [str(p) for p in CKPTS],
    "dataset_issues": summary["issues"],
    "padding": summary["padding"],
    "load_report": load_report,
}
json.dump(report, open(os.path.join(OUT_DIR, "inspection_report.json"), "w"), indent=1, default=str)
print("Saved to", OUT_DIR, ":", sorted(os.listdir(OUT_DIR)))